# Predicting apartment prices in Mexico City

## Objective
To determine whether apartment prices differ meaningfully by borough in Mexico City's Distrito Federal and build a baseline model to predict price from property size and location.

## Data & Cleaning
Combined five raw csv files (~900 properties after cleaning) via a reusable `wrangle()` function in `src/housing/wrangle.py`, covered by a pytest suite in `tests/test_wrangle.py`.

Cleaning steps: filtered to Distrito Federal apartments under $100,000, trimmed the top/bottom 10% of surface area as outliers, extracted `borough` from a nested location string, split combined `lat-lon` into numeric coordinates and dropped high-null, leakage-prone and multicollinear columns (kept `surface_covered_in_m2`, dropped `surface_total_in_m2`).

## Borough Price Differences
- Price distributions are **left-skewed** in 13 of 15 boroughs (2 exceptions: Iztapalapa and Milpa Alta, both right-skewed), with a **5.1x variance ratio** across boroughs. Both violate ANOVA's assumptions.
- **Kruskal-Wallis** (H = 191.3, p = 3.21e-33) rejected the null hypothesis that all boroughs share the same price distribution.
- **Dunn's post-hoc test** (Holm-corrected, 105 pairwise comparisons) visualized as a significance heatmap rather than a raw table, given the volume of comparisons. Milpa Alta, Tlahuac, and Iztapalapa stand out with significant differences against nearly every other borough.
- **Effect size (ε² = 0.214)** - moderate.

## Baseline Models
Two models were compared against a naive median baseline, using a consistent 80/20 train/test split throughout:

| Model | Features | MAE | R² |
|---|---|---|---|
| Naive baseline | — | $15,906 | — |
| Model 1 | `surface_covered_in_m2` | $14,362 | 0.123 |
| Model 2 | `surface_covered_in_m2` + `borough` (one-hot) | **$11,312** | **0.430** |

Adding `borough` more than **triples** the variance explained (R² 0.123 → 0.430) and cuts error by a further ~21% beyond the area-only model.

### A bug worth documenting
An early version of Model 2 applied `sklearn`'s `OneHotEncoder` directly to a two-column DataFrame containing both the categorical `borough` and the numeric `surface_covered_in_m2`. Unlike `category_encoders.OneHotEncoder` (which auto-detects categorical columns), `sklearn`'s version encodes every column it's given, silently shredding the numeric area feature into ~370 near-duplicate dummy columns, one per distinct value. The model still ran and still beat Model 1, just by a suspiciously small margin (MAE 14,156 vs. the true 11,312), which is what prompted a closer look. The fix was to route `borough` through `OneHotEncoder` inside a `ColumnTransformer` and pass `surface_covered_in_m2` through unchanged. This is a useful cautionary example: broken feature encoding doesn't always throw an error. It can just quietly produce a worse, still plausible looking number.

## Conclusion & Next Steps
Borough is not just statistically significant, it is the primary practical driver of price in this dataset. A natural next step is testing whether adding `lat`/`lon` directly (rather than only categorical borough) captures finer grained location effects within boroughs, or whether interaction terms between area and borough (different price-per-m² by borough) improve on the current additive model.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import kruskal
import scikit_posthocs as sp
import scipy.stats as stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

import sys
import os

# Adds the project root directory to the python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Now your import will work perfectly
from src.housing.wrangle import wrangle

In [ ]:
# Wrangling all 5 datasets and concatenating them into one dataframe
df1 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_1.csv")
df2 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_2.csv")
df3 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_3.csv")
df4 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_4.csv")
df5 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_5.csv")

df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

df.head()

In [ ]:
# Scatter map plot showing the price of apartments in Mexico City
fig = px.scatter_map(
    df,
    lat = "lat",
    lon = "lon",
    color = "price_aprox_usd",
    center = {"lat": 19.43, "lon": -99.13},
    width = 600,
    height = 600
)

fig.update_layout(map_style = "open-street-map")

fig.show()

In [ ]:
# A histogram to show apartment price distribution
plt.hist(df["price_aprox_usd"])

plt.title("Price Distribution")
plt.xlabel("Price [USD]")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# A scatter plot of price and surface area covered
plt.scatter(df["price_aprox_usd"], df["surface_covered_in_m2"])

plt.xlabel("Price [USD]")
plt.ylabel("Surface covered [m2]")
plt.title("A scatter plot of price and the surface area covered")

plt.show()

In [ ]:
# A bar graph showing the mean price of apartments by borough
mean_price_by_borough = df.groupby("borough")["price_aprox_usd"].mean().sort_values(ascending = True)

mean_price_by_borough.plot(
    kind = "bar",
    xlabel = "Borough",
    ylabel = "Price [USD]"
)

plt.title("A bar graph showing the mean price by borough")

plt.show()

In [ ]:
# Price distribution by borough
order = df.groupby("borough")["price_aprox_usd"].median().sort_values().index

plt.figure(figsize=(8, 8))

sns.boxplot(data=df, y="borough", x="price_aprox_usd", order=order)

plt.xlabel("Price [USD]")
plt.ylabel("Borough")
plt.title("Price Distribution by Borough")

plt.show()

In [ ]:
# Price skewness and variance by borough
print(df.groupby("borough")["price_aprox_usd"].skew().sort_values())
print(df.groupby("borough")["price_aprox_usd"].var().sort_values())

In [ ]:
# Histograms of price distribution by borough
boroughs = sorted(df['borough'].unique())
fig, axes = plt.subplots(len(boroughs), 1, figsize=(8, 4 * len(boroughs)))

for i, borough in enumerate(boroughs):
    subset = df[df['borough'] == borough]
    axes[i].hist(subset['price_aprox_usd'], bins=30, color='skyblue', edgecolor='black')
    axes[i].set_title(f'Price Distribution in {borough}')
    axes[i].set_xlabel('Price [USD]')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Kruskal-Wallis test across boroughs
groups = [g["price_aprox_usd"].values for _, g in df.groupby("borough")]
stat, p_value = kruskal(*groups)
print("Hstat:", f"{stat:.2e}")
print("P value:", f"{p_value:.2e}")

In [ ]:
# Compute epsilon-squared
n = len(df)
epsilon_sq = stat / (n - 1)
print(f"epsilon^2 = {epsilon_sq:.3f}")

In [ ]:
# Run Dunn's Test
dunn_results = sp.posthoc_dunn(
    df,
    val_col = "price_aprox_usd",
    group_col = "borough",
    p_adjust = "holm"
)

print(dunn_results)

In [ ]:
# Statistically significant pairwise borough price differences (Visualizing Holm-corrected Dunn's test pairwise results)
alpha = 0.05
significant = dunn_results < alpha

plt.figure(figsize=(10, 8))
sns.heatmap(
    significant, 
    cmap=["lightgray", "crimson"],
    cbar=False,
    linewidths=0.5,
    linecolor="white"
)

plt.title("Statistically Significant Pairwise Borough Price Differences\n(Dunn's Test, Holm-corrected, α=0.05)")
plt.show()

In [ ]:
# Correlation between "surface_covered_in_m2" and "price_aprox_usd" by borough
borough_corr = df.groupby("borough")[["surface_covered_in_m2", "price_aprox_usd"]].apply(
    lambda g: g["surface_covered_in_m2"].corr(g["price_aprox_usd"])
).sort_values()

sns.heatmap(
    borough_corr.to_frame(),
    annot = True,
    cmap = "coolwarm",
    vmin = -1,
    vmax = 1,
    fmt = ".2f"
)

plt.title("Surface Area–Price Correlation by Borough")

plt.show()

In [ ]:
# Setting the features and target for predicting the apartment price by "surface_covered_in_m2"
features_1 = ["surface_covered_in_m2"]
target = "price_aprox_usd"

X_1 = df[features_1]
y = df[target]

# Split the dataframe into 80% train data and 20% test data
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_1, y, test_size = 0.2, random_state = 42)

In [ ]:
# Naive baseline prediction on training data
baseline_pred_test = y_train_1.median()
mae_baseline_test = mean_absolute_error(y_test_1, [baseline_pred_test] * len(y_test_1))
print("Baseline Mean absolute error:", round(mae_baseline_test,2))

In [ ]:
# Model 1 using "surface_covered_in_m2" as the only feature
ridge_1 = Ridge()
ridge_1.fit(X_train_1, y_train_1)

print("Slope:", ridge_1.coef_[0])
print("Intercept:", ridge_1.intercept_)

In [ ]:
# Model 1 prediction
y_pred_1 = ridge_1.predict(X_test_1)

In [ ]:
# Model 1 mean absolute error
mae_1 = mean_absolute_error(y_test_1, y_pred_1)
print("Mean absolute error for model 1:", round(mae_1, 2))

In [ ]:
# Model 1 R2 score
R2_1 = ridge_1.score(X_test_1, y_test_1)
print("R2 score for Model 1:", round(R2_1,5))

In [ ]:
# Model 2 features ("surface_covered_in_m2", "borough")
features_2 = ["surface_covered_in_m2", "borough"]

X_2 = df[features_2]

# Train test split for model 2 into 80% train data and 20% test data
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_2, y, test_size = 0.2, random_state = 42)

In [ ]:
# OneHotEncoder on the "borough" column only
preprocessor = ColumnTransformer(
    [
        ("borough_ohe",
         OneHotEncoder(handle_unknown="ignore"),
         ["borough"]
         )
    ],
    remainder="passthrough"
)

# Model 2 pipeline including preprocessor and Ridge
model_2 = make_pipeline(
    preprocessor,
    Ridge()
)

# Fit model
model_2.fit(X_train_2, y_train_2)

In [ ]:
# Model 2 prediction
y_pred_2 = model_2.predict(X_test_2)

In [ ]:
# Model 2 mean absolute error
mae_2 = mean_absolute_error(y_test_2, y_pred_2)
print("Mean absolute error for model 2:", round(mae_2, 2))

In [ ]:
# Feature importances
coefficients = model_2.named_steps["ridge"].coef_
features = model_2.named_steps["columntransformer"].get_feature_names_out()
feat_imp = pd.Series(coefficients, index = features).sort_values(key = abs)
feat_imp

In [ ]:
feat_imp.plot(
    kind = "barh",
    x = "coefficients",
    y = "features"
)

plt.xlabel("Importance [USD]")
plt.ylabel("Feature")
plt.title("Feature Importances for apartment price")

plt.show()

In [ ]:
# Model 2 R2 score
R2_2 = model_2.score(X_test_2, y_test_2)
print("R2 score for Model 2:", round(R2_2,5))